In [ ]:
#08/16/2025

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import time

# %load_ext autoreload
# %autoreload 2

In [ ]:

from sian.data import final_save_header
from sian.data import final_save_dataset
from sian.data import final_save_labels
from sian.utils import gettimestamp

from sian.synth import Discrete_HierarchicalNoiseModel_InputDataGenerator
from sian.synth import DiscreteToContinuous_PurifiedCoefficients3D_OutputDataFunction


base_save_dataset_path = "data/"
datetimestr = gettimestamp()

In [ ]:
TRN_N = 1000*1000
TST_N = 10*1000
N = TRN_N + TST_N

In [ ]:
# Ds = [10,20,30]
# Ds = [8, 10, 12, 15] 
# Ds = [10]
Ds = [9]
list_of_max_tuples_per_band = []
for s in [3,4,5]:
    max_dict_v1 = {}
    max_dict_v2 = {}
    for k in range(1,3+1):
        max_dict_v1[k] = s**k
        max_dict_v2[k] = s
    list_of_max_tuples_per_band.append(max_dict_v1)
    list_of_max_tuples_per_band.append(max_dict_v2)
# all_I_ks = [4, 5, 6]
all_I_ks = [5, 10]
seeds=list(range(5))
seeds=list(range(1))

print('list_of_max_tuples_per_band')
print(list_of_max_tuples_per_band)


band_strengths = {
    1 : 1/3,
    2 : 1/3,
    3 : 1/3,
}


# anova_purification_type = "purification_off"
# anova_purification_type = "marginal_ANOVA"
# anova_purification_type = "conditional_Sobol_ANOVA"
# anova_purification_type = "conditional_Hooker_ANOVA"
anova_purification_type_list = ["marginal_ANOVA", "conditional_Sobol_ANOVA", "conditional_Hooker_ANOVA", ]

In [ ]:
SAVING = False
SAVING = True

def convert_to_saveable_dataset(fullX, fullY, D, I_k, fake_header_dict):
    pass

    readable_labels = {}
    full_readable_labels = {}
    startdim = 0
    for d in range(-1,D):
        full_info_d = {
            "label" : "X"+str(d+1),
            "startdim" : d*I_k,
            "numdims" : I_k,
            "count" : I_k,
            "encoding" : "disc.onehot",
        }
        if d==-1:
            full_info_d = {
                "label" : "Y",
                "startdim" : 0,
                "numdims" : 1,
                "encoding" : "cts.raw",
            }
        readable_labels[d] = full_info_d["label"]
        full_readable_labels[d] = full_info_d
    full_readable_labels["task_type"] = "regression"
    full_readable_labels["D0"] = D
    full_readable_labels["D"] = D


    trainval_portion, is_test_split_shuffled, shuffle_test_split_seed = None, False, None
    if trainval_portion is None:
        trainval_portion = 0.80
        trainval_portion = TRN_N/N
        assert int(trainval_portion*N)==TRN_N
    fake_header_dict["is_test_split_shuffled"]  = is_test_split_shuffled
    fake_header_dict["shuffle_test_split_seed"] = shuffle_test_split_seed
    fake_header_dict["trainval_portion"]        = trainval_portion
    

    label_stuff = (readable_labels, full_readable_labels)    
    label_dict = {
        "readable_labels" : label_stuff[0],
        "full_readable_labels" : label_stuff[1],
    }

    if SAVING:
        final_save_header(save_dataset_path, fake_header_dict)
        trnvalX,trnvalY,tstX,tstY = final_save_dataset(save_dataset_path, fullX, fullY, 
                                                        trainval_portion, is_test_split_shuffled, shuffle_test_split_seed)
        ### #### #### #### ###
        final_save_labels(save_dataset_path, label_dict)

In [ ]:

for D in Ds:
    for I_k in all_I_ks:
        for max_tuples_per_band in list_of_max_tuples_per_band:
            for seed in seeds:
                
                for purif_type in anova_purification_type_list:
                    for hierarchy_depth in [0,1,2]:
                        eps = 0.5
                        correlated_branching = 3
                        original_sources = D // (correlated_branching**hierarchy_depth)
                        print('hierarchy_depth',hierarchy_depth)
                        print('correlated_branching',correlated_branching)
                        print('original_sources',original_sources)
                        print('D',D)
                        print()

                        np.random.seed(seed)
                        band_sparsity_str = "s" + ".".join([str(max_tuples_per_band[k+1]) for k in range(3)])
                        purif_type_str = purif_type.replace("_","")
                        synth_dataset_id = f"SYNTH_simple_discrete_synthwave_v11_D{D}_Ik{I_k}_depth{hierarchy_depth}_{band_sparsity_str}_{purif_type_str}_seed{seed}"
                        preproc_owner = None
                        print(synth_dataset_id)
                        save_dataset_path = base_save_dataset_path + synth_dataset_id + "/"
                        fake_header_dict = {
                            "Preprocessed Datetime" : datetimestr, 
                            "dataset_id" : synth_dataset_id,
                            "preproc_owner" : preproc_owner,
                            "datagen_seed" : seed,
                        }


                        # dgp_x = Discrete_HierarchicalNoiseModel_InputDataGenerator(I_k, eps=0.0, sources=D, branching=0, depth=0)
                        # dgp_x = Discrete_HierarchicalNoiseModel_InputDataGenerator(I_k, eps=0.50, sources=1, branching=D, depth=1)
                        dgp_x = Discrete_HierarchicalNoiseModel_InputDataGenerator(I_k, eps=0.50, sources=original_sources, branching=correlated_branching, depth=hierarchy_depth)
                        dgp_y = DiscreteToContinuous_PurifiedCoefficients3D_OutputDataFunction(D, I_k, max_tuples_per_band, band_strengths, purif_type, dgp_x)

                        fake_header_dict["dgp_x_meta_hypers"] = dgp_x.generator_parameters
                        fake_header_dict["dgp_y_meta_hypers"] = dgp_y.generator_meta_parameters
                        print(fake_header_dict)
                        
                        X_dict = dgp_x.generate_samples(N)
                        X_onehot = np.concatenate([X_dict[d] for d in range(D)],axis=1)
                        y = dgp_y.generate_output_samples(X_onehot)

                        pass

                        convert_to_saveable_dataset(X_onehot, y,     D, I_k, fake_header_dict)
